# Divergence Detection Tests - Real BTC Data

This notebook tests divergence detection on **real Bitcoin 5m data**:
- Regular Bullish: Price LL + Indicator HL (reversal up)
- Regular Bearish: Price HH + Indicator LH (reversal down)
- Hidden Bullish: Price HL + Indicator LL (continuation up)
- Hidden Bearish: Price LH + Indicator HH (continuation down)

**Test Strategy**: 16 hand-picked ranges from BTC history (2 per divergence type)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from utils import add_zigzag, add_indicators, add_divergence

# Load BTC data
print("Loading BTC 5m data...")
df = pd.read_pickle('data/binance-BTCUSDT-5m.pkl')
df['date'] = pd.to_datetime(df['date_close'])

print("Adding technical indicators...")
add_indicators(df)

print("Adding zigzag columns...")
add_zigzag(df, threshold=0.1, column_name='zigzag_0_1')
add_zigzag(df, threshold=0.5, column_name='zigzag_0_2')
add_zigzag(df, threshold=1, column_name='zigzag_0_3')
add_zigzag(df, threshold=2, column_name='zigzag_0_5')
add_zigzag(df, threshold=5, column_name='zigzag_1')

print("Precomputing divergences...")
# Add divergence columns for RSI
add_divergence(
    df,
    indicator_column='rsi',
    zigzag_columns=['zigzag_0_1', 'zigzag_0_2', 'zigzag_0_3', 'zigzag_0_5', 'zigzag_1'],
    price_column='close'
)

# Add divergence columns for MACD
add_divergence(
    df,
    indicator_column='macd',
    zigzag_columns=['zigzag_0_1', 'zigzag_0_2', 'zigzag_0_3', 'zigzag_0_5', 'zigzag_1'],
    price_column='close'
)

df = df.dropna().reset_index(drop=True)

print(f"✓ Loaded {len(df):,} candles")
print(f"  Date range: {df['date'].iloc[0]} to {df['date'].iloc[-1]}")
print(f"  Available indicators: {[col for col in df.columns if col in ['rsi', 'macd', 'macd_signal', 'macd_hist']]}")
print(f"  Divergence columns: {[col for col in df.columns if '_div_' in col]}")

# Print divergence statistics
print("\n" + "="*80)
print("DIVERGENCE STATISTICS")
print("="*80)
div_cols = [col for col in df.columns if '_div_' in col]
for col in div_cols:
    non_zero = (df[col] != 0).sum()
    print(f"  {col}: {non_zero} divergences")

Loading BTC 5m data...
Adding technical indicators...
Adding zigzag columns...
Adding zigzag columns...
Precomputing divergences...
Precomputing divergences...
✓ Loaded 264,224 candles
  Date range: 2023-04-25 07:25:00 to 2025-10-28 18:00:00
  Available indicators: ['rsi', 'macd', 'macd_signal', 'macd_hist']
  Divergence columns: ['rsi_div_zigzag_0_1', 'rsi_div_zigzag_0_2', 'rsi_div_zigzag_0_3', 'rsi_div_zigzag_0_5', 'rsi_div_zigzag_1', 'macd_div_zigzag_0_1', 'macd_div_zigzag_0_2', 'macd_div_zigzag_0_3', 'macd_div_zigzag_0_5', 'macd_div_zigzag_1']

DIVERGENCE STATISTICS
  rsi_div_zigzag_0_1: 39283 divergences
  rsi_div_zigzag_0_2: 9884 divergences
  rsi_div_zigzag_0_3: 3426 divergences
  rsi_div_zigzag_0_5: 952 divergences
  rsi_div_zigzag_1: 168 divergences
  macd_div_zigzag_0_1: 46430 divergences
  macd_div_zigzag_0_2: 9303 divergences
  macd_div_zigzag_0_3: 3166 divergences
  macd_div_zigzag_0_5: 931 divergences
  macd_div_zigzag_1: 177 divergences
✓ Loaded 264,224 candles
  Date ra

In [2]:
# Simple Divergence Viewer - uses precomputed DataFrame columns
def plot_divergences_from_df(df_slice, start_idx, zigzag_col='zigzag_1', indicator='rsi', filter_type='All'):
    """Plot candlesticks with divergences from precomputed DataFrame columns."""
    # Close any existing figures to avoid duplicates
    plt.close('all')
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    
    # Plot candlesticks
    for i in range(len(df_slice)):
        row = df_slice.iloc[i]
        o, h, l, c = row['open'], row['high'], row['low'], row['close']
        color = 'green' if c >= o else 'red'
        
        # Wick
        ax1.plot([i, i], [l, h], color=color, linewidth=1, zorder=1)
        
        # Body
        height = abs(c - o)
        bottom = min(o, c)
        rect = Rectangle((i - 0.3, bottom), 0.6, height, 
                         facecolor=color, edgecolor=color, alpha=0.8, zorder=2)
        ax1.add_patch(rect)
    
    # Filter mapping
    filter_map = {
        'All': [1, 2, 3, 4],
        'Regular Bullish': [1],
        'Regular Bearish': [2],
        'Hidden Bullish': [3],
        'Hidden Bearish': [4]
    }
    allowed_types = filter_map.get(filter_type, [1, 2, 3, 4])
    
    # Mark divergences from precomputed column
    div_col = f'{indicator}_div_{zigzag_col}'
    if div_col in df_slice.columns:
        div_colors = {1: 'cyan', 2: 'magenta', 3: 'lime', 4: 'orange'}
        div_names = {1: 'REG BULL', 2: 'REG BEAR', 3: 'HID BULL', 4: 'HID BEAR'}
        
        for i, row in df_slice.iterrows():
            div_val = row[div_col]
            if div_val != 0 and div_val in allowed_types:
                rel_idx = i - df_slice.index[0]
                # Position marker based on divergence type
                if div_val in [2, 4]:  # Bearish - at top
                    price = row['high'] * 1.001
                    marker = 'v'
                    va = 'top'
                else:  # Bullish - at bottom
                    price = row['low'] * 0.999
                    marker = '^'
                    va = 'bottom'
                
                ax1.scatter(rel_idx, price, marker=marker, color=div_colors[div_val], 
                           s=250, zorder=7, edgecolors='black', linewidth=2.5)
                ax1.text(rel_idx, price, div_names[div_val], fontsize=9, ha='center',
                        va=va, weight='bold', color='white',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor=div_colors[div_val], 
                                 edgecolor='black', linewidth=1.5, alpha=0.9))
    
    title_suffix = f" - Filter: {filter_type}" if filter_type != 'All' else ""
    ax1.set_ylabel('Price', fontsize=12, fontweight='bold')
    ax1.grid(alpha=0.3)
    ax1.set_title(f'Divergences from {div_col}{title_suffix}', fontsize=14, fontweight='bold')
    
    # Plot indicator
    x_range = np.arange(len(df_slice))
    ax2.plot(x_range, df_slice[indicator].values, 'orange', label=indicator.upper(), linewidth=1.5)
    
    # Mark divergence points on indicator too
    if div_col in df_slice.columns:
        for i, row in df_slice.iterrows():
            div_val = row[div_col]
            if div_val != 0 and div_val in allowed_types:
                rel_idx = i - df_slice.index[0]
                ind_val = row[indicator]
                marker = 'v' if div_val in [2, 4] else '^'
                ax2.scatter(rel_idx, ind_val, marker=marker, color=div_colors[div_val],
                           s=150, zorder=7, edgecolors='black', linewidth=2)
    
    ax2.set_ylabel(indicator.upper(), fontsize=12, fontweight='bold')
    ax2.legend(loc='upper left', fontsize=10)
    ax2.grid(alpha=0.3)
    ax2.set_xlabel('Relative Index', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Print divergence summary
    divs_found = df_slice[df_slice[div_col] != 0]
    if filter_type != 'All':
        divs_found = divs_found[divs_found[div_col].isin(allowed_types)]
    
    if len(divs_found) > 0:
        print(f"\nFound {len(divs_found)} divergences in range:")
        div_type_names = {1: 'Regular Bullish', 2: 'Regular Bearish', 
                         3: 'Hidden Bullish', 4: 'Hidden Bearish'}
        for idx, row in divs_found.iterrows():
            div_type = div_type_names[row[div_col]]
            print(f"  Index {idx} (rel {idx - df_slice.index[0]}): {div_type} | Price: ${row['close']:.2f} | {indicator.upper()}: {row[indicator]:.2f}")
    else:
        filter_msg = f" (filter: {filter_type})" if filter_type != 'All' else ""
        print(f"\nNo divergences found in this range for {div_col}{filter_msg}")

print("✓ Divergence viewer function loaded - use the interactive navigator below!")

✓ Divergence viewer function loaded - use the interactive navigator below!


In [3]:
# Interactive Navigator - uses precomputed divergences from DataFrame
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuration
MAX_INDEX = len(df) - 500

# Interactive state
current_state = {
    'start_idx': 10000,
    'window_size': 200,
    'zigzag_col': 'zigzag_1',
    'indicator': 'rsi',
    'filter_type': 'All'  # Filter for divergence types
}

def analyze_and_plot_precomputed(start_idx, window_size, zigzag_col, indicator, filter_type='All'):
    """Plot divergences from precomputed DataFrame columns."""
    with output_area:
        clear_output(wait=True)
        
        # Validate range
        end_idx = min(start_idx + window_size, len(df))
        start_idx = max(0, start_idx)
        
        # Extract data slice
        df_slice = df.iloc[start_idx:end_idx]
        
        # Plot using precomputed data with filter
        plot_divergences_from_df(df_slice, start_idx, zigzag_col=zigzag_col, indicator=indicator, filter_type=filter_type)
        
        # Print analysis
        print(f"\n{'='*80}")
        print(f"RANGE: {start_idx:,} to {end_idx:,} ({window_size} candles)")
        print(f"{'='*80}")
        print(f"Date range: {df_slice['date'].iloc[0]} to {df_slice['date'].iloc[-1]}")
        print(f"Price range: ${df_slice['low'].min():.2f} - ${df_slice['high'].max():.2f}")
        print(f"{indicator.upper()} range: {df_slice[indicator].min():.1f} - {df_slice[indicator].max():.1f}")

def on_slider_change(change):
    """Handle slider value changes."""
    current_state['start_idx'] = change['new']

def on_window_change(change):
    """Handle window size changes."""
    current_state['window_size'] = change['new']

def on_zigzag_change(change):
    """Handle zigzag column changes."""
    current_state['zigzag_col'] = change['new']

def on_indicator_change(change):
    """Handle indicator changes."""
    current_state['indicator'] = change['new']

def on_filter_change(change):
    """Handle filter changes."""
    current_state['filter_type'] = change['new']

def on_update_click(b):
    """Handle update button click."""
    analyze_and_plot_precomputed(
        current_state['start_idx'], 
        current_state['window_size'],
        current_state['zigzag_col'],
        current_state['indicator'],
        current_state['filter_type']
    )

def on_previous_click(b):
    """Navigate to previous range by 1 step."""
    step = 1  # Move by 1 candle
    current_state['start_idx'] = max(0, current_state['start_idx'] - step)
    start_slider.value = current_state['start_idx']
    analyze_and_plot_precomputed(
        current_state['start_idx'], 
        current_state['window_size'],
        current_state['zigzag_col'],
        current_state['indicator'],
        current_state['filter_type']
    )

def on_next_click(b):
    """Navigate to next range by 1 step."""
    step = 1  # Move by 1 candle
    current_state['start_idx'] = min(MAX_INDEX, current_state['start_idx'] + step)
    start_slider.value = current_state['start_idx']
    analyze_and_plot_precomputed(
        current_state['start_idx'], 
        current_state['window_size'],
        current_state['zigzag_col'],
        current_state['indicator'],
        current_state['filter_type']
    )

def on_show_all_click(b):
    """Show all divergences in the dataset."""
    with output_area:
        clear_output(wait=True)
        div_col = f"{current_state['indicator']}_div_{current_state['zigzag_col']}"
        
        # Find all divergences
        all_divs = df[df[div_col] != 0].copy()
        
        # Apply filter if needed
        filter_type = current_state['filter_type']
        if filter_type != 'All':
            filter_map = {
                'Regular Bullish': 1,
                'Regular Bearish': 2,
                'Hidden Bullish': 3,
                'Hidden Bearish': 4
            }
            all_divs = all_divs[all_divs[div_col] == filter_map[filter_type]]
        
        div_type_names = {1: 'Regular Bullish', 2: 'Regular Bearish', 
                         3: 'Hidden Bullish', 4: 'Hidden Bearish'}
        
        print(f"{'='*80}")
        print(f"ALL DIVERGENCES: {div_col}")
        if filter_type != 'All':
            print(f"Filter: {filter_type}")
        print(f"{'='*80}")
        print(f"Total found: {len(all_divs)}\n")
        
        # Group by type
        for div_type in [1, 2, 3, 4]:
            divs_of_type = all_divs[all_divs[div_col] == div_type]
            if len(divs_of_type) > 0:
                print(f"\n{div_type_names[div_type]}: {len(divs_of_type)} occurrences")
                print("-" * 80)
                for idx, row in divs_of_type.head(20).iterrows():
                    print(f"  Index {idx:6d} | Date: {row['date']} | Price: ${row['close']:8.2f} | {current_state['indicator'].upper()}: {row[current_state['indicator']]:6.2f}")
                if len(divs_of_type) > 20:
                    print(f"  ... and {len(divs_of_type) - 20} more")

def on_jump_click(b):
    """Jump to specific preset location."""
    jump_locations = {
        'Start': 1000,
        'Early': 10000,
        'Mid-Early': 25000,
        'Middle': 50000,
        'Mid-Late': 75000,
        'Late': 100000,
        'Recent': max(0, len(df) - 5000)
    }
    location = jump_locations.get(jump_dropdown.value, 10000)
    current_state['start_idx'] = location
    start_slider.value = location
    analyze_and_plot_precomputed(
        current_state['start_idx'], 
        current_state['window_size'],
        current_state['zigzag_col'],
        current_state['indicator'],
        current_state['filter_type']
    )

# Create widgets
start_slider = widgets.IntSlider(
    value=current_state['start_idx'],
    min=0,
    max=MAX_INDEX,
    step=1,  # Changed to 1 for fine control
    description='Start Index:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px'),
    continuous_update=False
)

window_slider = widgets.IntSlider(
    value=current_state['window_size'],
    min=100,
    max=500,
    step=50,
    description='Window Size:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px'),
    continuous_update=False
)

zigzag_dropdown = widgets.Dropdown(
    options=['zigzag_0_1', 'zigzag_0_2', 'zigzag_0_3', 'zigzag_0_5', 'zigzag_1'],
    value='zigzag_1',
    description='ZigZag:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='250px')
)

indicator_dropdown = widgets.Dropdown(
    options=['rsi', 'macd'],
    value='rsi',
    description='Indicator:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='200px')
)

filter_dropdown = widgets.Dropdown(
    options=['All', 'Regular Bullish', 'Regular Bearish', 'Hidden Bullish', 'Hidden Bearish'],
    value='All',
    description='Filter:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='250px')
)

update_button = widgets.Button(
    description='🔄 Update',
    button_style='primary',
    layout=widgets.Layout(width='120px')
)

previous_button = widgets.Button(
    description='◀ Previous',
    button_style='info',
    layout=widgets.Layout(width='120px')
)

next_button = widgets.Button(
    description='Next ▶',
    button_style='info',
    layout=widgets.Layout(width='120px')
)

show_all_button = widgets.Button(
    description='📋 Show All',
    button_style='warning',
    layout=widgets.Layout(width='120px')
)

jump_dropdown = widgets.Dropdown(
    options=['Start', 'Early', 'Mid-Early', 'Middle', 'Mid-Late', 'Late', 'Recent'],
    value='Early',
    description='Jump to:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='200px')
)

jump_button = widgets.Button(
    description='🎯 Jump',
    button_style='success',
    layout=widgets.Layout(width='100px')
)

# Wire up callbacks
start_slider.observe(on_slider_change, names='value')
window_slider.observe(on_window_change, names='value')
zigzag_dropdown.observe(on_zigzag_change, names='value')
indicator_dropdown.observe(on_indicator_change, names='value')
filter_dropdown.observe(on_filter_change, names='value')
update_button.on_click(on_update_click)
previous_button.on_click(on_previous_click)
next_button.on_click(on_next_click)
show_all_button.on_click(on_show_all_click)
jump_button.on_click(on_jump_click)

# Create output area
output_area = widgets.Output()

# Layout
controls = widgets.VBox([
    widgets.HTML("<h3>📊 Interactive Divergence Explorer (Precomputed Data)</h3>"),
    start_slider,
    window_slider,
    widgets.HBox([zigzag_dropdown, indicator_dropdown, filter_dropdown]),
    widgets.HBox([previous_button, next_button, update_button, show_all_button]),
    widgets.HBox([jump_dropdown, jump_button]),
    widgets.HTML("<hr>"),
    output_area
])

display(controls)

print("\n✓ Interactive navigator loaded - uses precomputed divergences from DataFrame!")
print(f"✓ Total data available: {len(df):,} candles")
print(f"✓ Date range: {df['date'].iloc[0]} to {df['date'].iloc[-1]}")
print("\n👉 Use 'Filter' dropdown to show specific divergence types")
print("👉 Click '📋 Show All' to list all divergences in the dataset")
print("👉 ◀/▶ buttons move by 1 candle for precise navigation")


✓ Interactive navigator loaded - uses precomputed divergences from DataFrame!
✓ Total data available: 264,224 candles
✓ Date range: 2023-04-25 07:25:00 to 2025-10-28 18:00:00

👉 Use 'Filter' dropdown to show specific divergence types
👉 Click '📋 Show All' to list all divergences in the dataset
👉 ◀/▶ buttons move by 1 candle for precise navigation
